In [ ]:

import numpy as np
import os
import pandas as pd
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from core.Log import *
from core.plots import *
from core.model_utils import *
from core.CNNmodel import *
from core.benchmarks import MetadataMLP
from core.CardiacCTdataset import DataLoaderFactory
from core.preprocessing import crop_training
from core.globals import *
import logging
from core.CVsplits import *
from tqdm.notebook import tqdm


pools = ["holdout", "main"]

OUTER_FOLDS = 4; INNER_FOLDS = 3

main_dataset = load_dataset_info(file="data/data_info.json")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
crop_training(main_dataset)


Cropping training data: 100%|██████████| 157/157 [09:01<00:00,  3.45s/it]


In [24]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",False,False
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",False,False


In [ ]:
log = logging.getLogger('OUTER_train')
log.info(f"		 	 Model;		ExpID;	OUT;	 HPset;  	 		 Epoch; 		T_loss; 						T_acc;								V_loss; 	 				V_acc;		P; 		LR")
for experiment in final_experiment:
	trained = experiment["trained"]
	if trained: continue
	model_type = experiment["Model"]
	if not model_type == "SingleView_Sagittal": continue
	fold = experiment["OUTER_FOLD"]
	HPset = experiment["hypers"]["HPset"]
	train_loader, val_loader, _ = DL.create_outer_loaders(fold)
	DR = experiment['hypers']['DR']
	model=SingleViewClassifier(DR)
	model, best_model_state = train_SINGLEVIEW(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/{model_type}_fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")


	↳ Single View Experiment 9 | Training model... 
	↳ Single View Experiment 10 | Training model... 
	↳ Single View Experiment 11 | Training model... 
	↳ Single View Experiment 12 | Training model... 


In [ ]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",False,False
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False


In [ ]:
log = logging.getLogger('OUTER_train')
log.info(f"		 	 Model;		ExpID;	OUT;	 HPset;  	 		 Epoch; 		T_loss; 						T_acc;								V_loss; 	 				V_acc;		P; 		LR")
for experiment in final_experiment:
	trained = experiment["trained"]
	if trained: continue
	model_type = experiment["Model"]
	if not model_type == "MLP_META": continue
	fold = experiment["OUTER_FOLD"]
	HPset = experiment["hypers"]["HPset"]
	train_loader, val_loader, _ = DL.create_outer_loaders(fold)
	DR = experiment['hypers']['DR']
	model = MetadataMLP(DR)
	model, best_model_state = train_MLP(model, train_loader, val_loader, experiment)
	torch.save(best_model_state, f"pth_models/{model_type}_fold_{fold}_HPset_{HPset}.pth")
	experiment["trained"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")


	↳ Experiment 1 | Training model... 
	↳ Experiment 2 | Training model... 
	↳ Experiment 3 | Training model... 
	↳ Experiment 4 | Training model... 


## Outer Folds Evaluations

In [ ]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,False
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,False
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,False
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,False
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,False


In [ ]:
def EVALUATE_MODEL(model, test_loader, experiment):
	hypers = experiment['hypers']
	TH = hypers['TH']
	model_type = experiment["Model"]
	ExpID = experiment['ExpID']
	fold = experiment["OUTER_FOLD"]
	log = logging.getLogger('OUTER_evaluate')

	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

	model.eval()
	criterion = nn.BCEWithLogitsLoss()
	eval_N = len(test_loader.dataset)

	running_loss = 0.0
	all_predictions = []
	all_probabilities = []
	all_labels = []
	per_case_rows = []


	#with torch.no_grad():
	with torch.inference_mode():
		print(f"	↳ Experiment {ExpID} | {model_type} | Evaluating model... ")
		for batch in test_loader:
			axi = batch["axial_image"].to(device)
			cor = batch["coronal_image"].to(device)
			sag = batch["sagittal_image"].to(device)
			met = batch["meta"].to(device)
			lbl = batch["label"].to(device).float().unsqueeze(1)
			case_ids = batch["CaseID"]

			#axi_features = feature_extractor(axi)
			#cor_features = feature_extractor(cor)
			#sag_features = feature_extractor(sag)

			#combined_input = torch.cat([axi_features, cor_features, sag_features, met], dim=1)

			#logits = model(combined_input)
			#logits = model(sag, meta=met)
			#logits = model(cor, meta=met)
			#logits = model(axi, meta=met)
			logits = model(axi, sag, cor, met)
			#logits = model(meta=met)


			#loss = criterion(logits, lbl)
			loss = criterion(logits, lbl)
			running_loss += loss.item() * lbl.size(0)


			probability = torch.sigmoid(logits)
			prediction = (probability >= TH).to(torch.int32)


			prob_np = probability.squeeze(1).cpu().numpy()
			pred_np = prediction.squeeze(1).cpu().numpy()
			lbl_np  = lbl.squeeze(1).cpu().numpy()

			all_probabilities.append(prob_np)
			all_predictions.append(pred_np)
			all_labels.append(lbl_np)


			#current_batch_size = len(case_ids)


			#for i in range(current_batch_size):

			for cid, y, yhat, p in zip(case_ids, lbl_np, pred_np, prob_np):
				per_case_rows.append({
					"Model": model_type,
					"ExpID": ExpID,
					"HPset": hypers["HPset"],
					"OUTER_FOLD": fold,
					"CaseID": cid,
					"Label": int(y),
					"Pred": int(yhat),
					"Prob": float(p),
					"TH_used": TH,
				})

				#case_id = CaseID[i]
				#label =  lbl[i].item()
				#pred = prediction[i].item()
				#prob = probability[i].item()
				log.info(f"   {model_type};  	{ExpID}; 	{hypers['HPset']}; 		{fold}; 	{cid}; 		{int(y)}		{int(yhat)};	 	{float(p):.4f};    {TH:.6f}")

	#all_probabilities = np.concatenate(all_probabilities, axis=0)
	#all_predictions = np.concatenate(all_predictions, axis=0)
	#all_labels = np.concatenate(all_labels, axis=0)
	final_loss = running_loss / eval_N
	return final_loss, per_case_rows


In [21]:
final_experiment = load_from_json(filename="NCV_4_3_folds/benchmark_experiments.json")
final_experiment_df = pd.DataFrame(final_experiment)
final_experiment_df


Loaded NCV_4_3_folds/benchmark_experiments.json.


,Model,ExpID,OUTER_FOLD,hypers,trained,evaluated
0,MLP_META,1,0,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,True
1,MLP_META,2,1,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,True
2,MLP_META,3,2,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,True
3,MLP_META,4,3,"{'HPset': 101, 'LR': 0.001, 'WD': 1e-05, 'DR':...",True,True
4,SingleView_Axial,5,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True
5,SingleView_Axial,6,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True
6,SingleView_Axial,7,2,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True
7,SingleView_Axial,8,3,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True
8,SingleView_Sagittal,9,0,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True
9,SingleView_Sagittal,10,1,"{'HPset': 201, 'LR': 0.0005, 'WD': 1e-05, 'DR'...",True,True


In [ ]:
log = logging.getLogger('OUTER_evaluate')
log.info("Time;		ExpID;  	HP_Set; 	Fold;		CaseID;   					GroundTruth;			prediction;		probability;")
for experiment in final_experiment:
	#print(experiment)
	evaluated = experiment["evaluated"]
	if evaluated: continue
	model_type = experiment["Model"]
	if not model_type == "ResNet18": continue
	fold = experiment["OUTER_FOLD"]
	HPset = experiment["hypers"]["HPset"]
	_, _, test_loader = DL.create_outer_loaders(fold)

	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	feature_extractor, model = create_adapted_resnet18(device)
	state_dict = torch.load(f"pth_models/{model_type}_fold_{fold}_HPset_{HPset}.pth")
	model.load_state_dict(state_dict)
	final_loss, all_probabilities, all_labels, all_predictions = EVALUATE_MODEL(feature_extractor, model, test_loader, experiment)
	experiment["evaluated"] = True
	save_to_json(final_experiment, filename="NCV_4_3_folds/benchmark_experiments.json")


	↳ Experiment 17 | ResNet18 | Evaluating model... 
	↳ Experiment 18 | ResNet18 | Evaluating model... 
	↳ Experiment 19 | ResNet18 | Evaluating model... 
	↳ Experiment 20 | ResNet18 | Evaluating model... 
